# **Sistem Antrian Rumah Sakit**

---


**Proyek Akhir Mata Kuliah:** Algoritma dan Pemrograman — Universitas Al Azhar Indonesia

## **Identitas**
- Kelompok: Sistem Antrian Rumah Sakit — Kelompok 5
- Nama Anggota:
  1. Ivanka Widyaningrum (0112525005)
  2. Lolita Disry Mustika (0112525006)
  3. Muhammad Fajri Febriyanto (0112525008)
  4. Sahid Haqqi An Najil (0112525013)
- Program Studi: Informatika
- Dosen: Tri Aji Nugroho, S.T., M.T.
- Semester: Genap 2025/2026


Notebook ini berisi kode lengkap sistem antrian rumah sakit, dengan **docstring format `Parameters`/`Returns` pada seluruh 27 fungsi** untuk keperluan dokumentasi kode.

Jalankan cell di bawah untuk memulai program (data demo 18 pasien akan otomatis dimuat).

In [3]:
"""
============================================================
SISTEM ANTRIAN RUMAH SAKIT / PUSKESMAS
Mata Kuliah: Algoritma dan Pemrograman
Universitas Al Azhar Indonesia
============================================================
"""

from datetime import datetime, timedelta

# ============================================================
# DATA GLOBAL
# ============================================================
data_pasien = {}              # key: no_RM -> dict biodata
antrian = {"darurat": [], "lansia": [], "umum": []}  # key: kategori -> list no_RM
riwayat_kunjungan = {}        # key: no_RM -> list of dict kunjungan
riwayat_durasi_layanan = []   # list durasi (menit) pasien yang sudah selesai dilayani
nomor_counter = 0

URUTAN_PRIORITAS = ["darurat", "lansia", "umum"]
WAKTU_DEFAULT = 5  # menit, dipakai jika belum ada riwayat durasi layanan


# ============================================================
# FUNGSI BANTU (VALIDASI & UTILITAS)
# ============================================================

def konfirmasi_yes_no(pertanyaan):
    """
    Meminta konfirmasi yes/no dari pengguna, hanya menerima 2 jawaban valid.
    Parameters:
        pertanyaan (str): Teks pertanyaan yang ditampilkan ke pengguna
    Returns:
        bool: True jika pengguna menjawab "yes", False jika menjawab "no"
    """
    while True:
        jawaban = input(pertanyaan).strip().lower()
        if jawaban == "yes":
            return True
        elif jawaban == "no":
            return False
        else:
            print("  [!] Input tidak valid. Ketik 'yes' atau 'no' saja.")


def label_kategori(kategori):
    """
    Mengubah nama kategori menjadi label tampilan yang mencolok.
    Parameters:
        kategori (str): Kode kategori ("darurat"/"lansia"/"umum")
    Returns:
        str: Label tampilan kategori, misalnya "[DARURAT]"
    """
    label = {"darurat": "[DARURAT]", "lansia": "[LANSIA]", "umum": "[UMUM]"}
    return label.get(kategori, "[UNKNOWN]")


def pilih_kategori():
    """
    Menampilkan pilihan kategori dan meminta petugas memilih secara manual.
    Parameters:
        Tidak ada.
    Returns:
        str: Kategori terpilih ("darurat", "lansia", atau "umum")
    """
    print("\nPilih kategori pasien:")
    print("  1. Darurat")
    print("  2. Lansia")
    print("  3. Umum")
    while True:
        pilihan = input("Masukkan pilihan (1-3): ").strip()
        if pilihan == "1":
            return "darurat"
        elif pilihan == "2":
            return "lansia"
        elif pilihan == "3":
            return "umum"
        else:
            print("  [!] Pilihan tidak valid, coba lagi.")


def validasi_nama(nama):
    """
    Memvalidasi nama pasien: mengecek apakah kosong, mengandung angka,
    atau mengandung karakter tidak umum.
    Parameters:
        nama (str): Nama pasien yang akan divalidasi
    Returns:
        tuple: (status (str), pesan (str)) -- status berupa salah satu dari
        "gagal" (input harus diulang), "peringatan" (perlu konfirmasi
        petugas), atau "valid" (nama diterima langsung)
    """
    nama = nama.strip()
    if nama == "":
        return "gagal", "Nama tidak boleh kosong."

    if any(karakter.isdigit() for karakter in nama):
        return "peringatan", f"Nama '{nama}' mengandung angka."

    karakter_diizinkan = set(
        "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ -'"
    )
    if not all(karakter in karakter_diizinkan for karakter in nama):
        return "peringatan", f"Nama '{nama}' mengandung karakter tidak umum."

    return "valid", ""


def input_nama():
    """
    Loop input nama dengan validasi dan konfirmasi jika ditemukan anomali.
    Parameters:
        Tidak ada.
    Returns:
        str: Nama pasien yang sudah tervalidasi/dikonfirmasi
    """
    while True:
        nama = input("Nama pasien: ").strip()
        status, pesan = validasi_nama(nama)

        if status == "gagal":
            print(f"  [!] {pesan}")
            continue

        if status == "peringatan":
            print(f"  [!] {pesan}")
            if not konfirmasi_yes_no(f"  Apakah nama '{nama}' sudah benar? (yes/no): "):
                continue

        return nama


def input_umur():
    """
    Loop input umur dengan validasi rentang wajar (1-120 tahun).
    Parameters:
        Tidak ada.
    Returns:
        int: Umur pasien (1-120) yang sudah tervalidasi
    """
    while True:
        teks_umur = input("Umur: ").strip()
        if not teks_umur.isdigit():
            print("  [!] Umur harus berupa angka.")
            continue
        umur = int(teks_umur)
        if umur <= 0 or umur > 120:
            print("  [!] Umur tidak valid (harus 1-120).")
            continue
        return umur


def input_nik():
    """
    Loop input NIK dengan validasi panjang 16 digit angka.
    Parameters:
        Tidak ada.
    Returns:
        str: NIK pasien (16 digit angka) yang sudah tervalidasi
    """
    while True:
        nik = input("NIK (16 digit): ").strip()
        if len(nik) != 16 or not nik.isdigit():
            print("  [!] NIK harus terdiri dari 16 digit angka.")
            continue
        return nik


def input_keluhan():
    """
    Loop input keluhan pasien, tidak boleh kosong.
    Parameters:
        Tidak ada.
    Returns:
        str: Keluhan pasien yang tidak kosong
    """
    while True:
        keluhan = input("Keluhan: ").strip()
        if keluhan == "":
            print("  [!] Keluhan tidak boleh kosong.")
            continue
        return keluhan


def buat_no_rm():
    """
    Membuat nomor Rekam Medis (RM) baru yang unik, format: RM001, RM002, dst.
    Hanya dipanggil untuk pasien BARU -- pasien lama menggunakan kembali
    no_rm yang sudah ada (lihat daftar_kunjungan_ulang()).
    Parameters:
        Tidak ada.
    Returns:
        str: Nomor RM baru, format "RM001", "RM002", dst.
    """
    global nomor_counter
    nomor_counter += 1
    return f"RM{nomor_counter:03d}"


# ============================================================
# DATA DEMO (dipakai setiap program dijalankan, untuk keperluan uji fitur 2-8)
# ============================================================

def muat_data_demo():
    """
    Memuat data pasien contoh ke dalam sistem, supaya fitur 2-8 bisa
    langsung diuji/didemokan tanpa harus mendaftar manual lewat fitur 1
    setiap kali program dijalankan.
    Parameters:
        Tidak ada.
    Returns:
        None: Tidak mengembalikan nilai -- langsung mengisi data_pasien,
        antrian, riwayat_kunjungan, dan riwayat_durasi_layanan secara global.
    """
    global nomor_counter
    sekarang = datetime.now()

    # (no_rm, nama, umur, nik, keluhan, kategori, menit_lalu_saat_daftar)
    contoh_pasien = [
        ("RM001", "Ivanka Widya",     34, "3201234567890001", "Kecelakaan lalu lintas", "darurat", 0),
        ("RM002", "Lolita Distri",    67, "3201234567890002", "Nyeri dada",             "lansia",  10),
        ("RM003", "Muhammad Fajri",     45, "3201234567890003", "Demam tinggi",           "umum",    15),
        ("RM004", "Sahid Haqqi",      72, "3201234567890004", "Nyeri sendi",            "lansia",  20),
        ("RM005", "Dedi Kurniawan",  28, "3201234567890005", "Sakit kepala",           "umum",    25),
        ("RM006", "Maya Putri",       8, "3201234567890006", "Batuk pilek",            "umum",    30),
        ("RM007", "Andi Wijaya",     55, "3201234567890007", "Sesak napas",            "darurat", 35),
        ("RM008", "Wulan Sari",      30, "3201234567890008", "Mual dan muntah",        "umum",    40),
        ("RM009", "Hendra Gunawan",  61, "3201234567890009", "Pusing berputar",        "lansia",  45),
        ("RM010", "Nurul Hidayah",   25, "3201234567890010", "Alergi kulit",           "umum",    50),
        ("RM011", "Joko Prasetyo",   47, "3201234567890011", "Nyeri punggung",         "umum",    55),
        ("RM012", "Kartika Dewi",    19, "3201234567890012", "Diare",                  "umum",    60),
        ("RM013", "Slamet Riyadi",   70, "3201234567890013", "Sesak napas ringan",     "lansia",  65),
        ("RM014", "Dewi Lestari",    33, "3201234567890014", "Luka ringan",            "umum",    70),
        ("RM015", "Fajar Nugroho",   42, "3201234567890015", "Pendarahan hebat",       "darurat", 75),
        ("RM016", "Rina Marlina",    65, "3201234567890016", "Demam dan lemas",        "lansia",  80),
        ("RM017", "Bayu Setiawan",   29, "3201234567890017", "Cedera olahraga",        "umum",    85),
        ("RM018", "Yuni Astuti",     52, "3201234567890018", "Vertigo",                "umum",    90),
    ]

    for no_rm, nama, umur, nik, keluhan, kategori, menit_lalu in contoh_pasien:
        data_pasien[no_rm] = {
            "nama": nama,
            "umur": umur,
            "nik": nik,
            "keluhan": keluhan,
            "kategori": kategori,
            "waktu_daftar": sekarang - timedelta(minutes=menit_lalu),
        }
        tambah_ke_antrian(no_rm, kategori)

    # Riwayat kunjungan sebelumnya -> untuk menguji fitur 5 (riwayat per pasien)
    tambah_riwayat_kunjungan("RM002", "Jantung", "Hipertensi terkontrol")
    tambah_riwayat_kunjungan("RM004", "Umum", "Cek rutin lansia")
    tambah_riwayat_kunjungan("RM004", "Jantung", "Nyeri sendi ringan")
    tambah_riwayat_kunjungan("RM009", "Saraf", "Vertigo ringan")
    tambah_riwayat_kunjungan("RM013", "Paru", "Riwayat asma")

    # Riwayat durasi layanan historis -> untuk menguji fitur 8 (estimasi waktu tunggu)
    riwayat_durasi_layanan.extend([8, 12, 6, 10, 15, 9, 11])

    # Lanjutkan penomoran RM baru dari RM019, bukan mengulang dari RM001
    nomor_counter = len(contoh_pasien)


# ============================================================
# MODUL PENDAFTARAN  (FR-1, FR-2, FR-10)
# ============================================================

def daftar_kunjungan_ulang(no_rm):
    """
    Mendaftarkan kunjungan baru untuk pasien LAMA (RM sudah ada), tanpa
    mengulang input biodata (nama/umur/NIK) yang sudah tercatat sebelumnya.
    Keluhan & kategori tetap ditanya karena bisa berbeda tiap kunjungan.
    NIK ditampilkan di ringkasan sebagai validasi tambahan -- membantu
    petugas memastikan identitas benar meskipun ada nama yang mirip/sama.
    Parameters:
        no_rm (str): Nomor RM pasien lama yang akan didaftarkan kunjungan
            ulangnya
    Returns:
        str atau None: no_rm jika kunjungan ulang berhasil didaftarkan,
        None jika pengguna membatalkan konfirmasi akhir
    """
    info = data_pasien[no_rm]
    print(f"\n[Ditemukan] {info['nama']}, {info['umur']} tahun (NIK: {info['nik']}).")
    tampilkan_riwayat_singkat(no_rm)

    keluhan = input_keluhan()
    kategori = pilih_kategori()

    print("\n--- RINGKASAN KUNJUNGAN ULANG ---")
    print(f"Nama     : {info['nama']}")
    print(f"No. RM   : {no_rm}")
    print(f"NIK      : {info['nik']}")
    print(f"Keluhan  : {keluhan}")
    print(f"Kategori : {kategori.capitalize()}")
    print("---------------------------------")

    if not konfirmasi_yes_no("Apakah data sudah sesuai? (yes/no): "):
        print("\n[X] Pendaftaran kunjungan ulang dibatalkan.")
        return None

    # Perbarui keluhan, kategori, dan waktu daftar untuk kunjungan kali ini.
    # Biodata (nama/umur/NIK) TIDAK diubah -- itu tetap data asli pasien.
    info["keluhan"] = keluhan
    info["kategori"] = kategori
    info["waktu_daftar"] = datetime.now()

    tambah_ke_antrian(no_rm, kategori)

    print(f"\n[OK] Kunjungan ulang berhasil didaftarkan. Nomor RM: {no_rm}")
    return no_rm


def daftar_pasien():
    """
    Mendaftarkan pasien ke antrian. Menanyakan dulu apakah pasien sudah
    pernah berobat sebelumnya (kunjungan ulang) atau pasien baru, supaya
    pasien lama tidak perlu mengulang input biodata lengkap. Pencarian
    pasien lama mendukung RM, NIK, atau nama (lihat cari_no_rm_pasien_lama).
    Parameters:
        Tidak ada.
    Returns:
        str atau None: Nomor RM pasien (baru maupun lama) jika pendaftaran
        berhasil, None jika dibatalkan pada salah satu tahap konfirmasi
    """
    print("\n=== PENDAFTARAN PASIEN ===")

    sudah_pernah = konfirmasi_yes_no(
        "Apakah pasien sudah pernah berobat di sini sebelumnya? (yes/no): "
    )

    if sudah_pernah:
        no_rm = cari_no_rm_pasien_lama()
        if no_rm is not None:
            return daftar_kunjungan_ulang(no_rm)

        if not konfirmasi_yes_no(
            "Lanjutkan sebagai pendaftaran pasien BARU? (yes/no): "
        ):
            print("\n[X] Pendaftaran dibatalkan.")
            return None
        # Nomor RM tidak ditemukan tapi pengguna memilih lanjut ->
        # alur di bawah (pasien baru) tetap dijalankan seperti biasa.

    print("\n--- DATA PASIEN BARU ---")
    nama = input_nama()
    umur = input_umur()
    nik = input_nik()
    keluhan = input_keluhan()
    kategori = pilih_kategori()

    print("\n--- RINGKASAN DATA ---")
    print(f"Nama     : {nama}")
    print(f"Umur     : {umur}")
    print(f"NIK      : {nik}")
    print(f"Keluhan  : {keluhan}")
    print(f"Kategori : {kategori.capitalize()}")
    print("-----------------------")

    if not konfirmasi_yes_no("Apakah data sudah sesuai? (yes/no): "):
        print("\n[X] Pendaftaran dibatalkan. Silakan ulangi.")
        return None

    no_rm = buat_no_rm()
    data_pasien[no_rm] = {
        "nama": nama,
        "umur": umur,
        "nik": nik,
        "keluhan": keluhan,
        "kategori": kategori,
        "waktu_daftar": datetime.now(),
    }

    tambah_ke_antrian(no_rm, kategori)

    print(f"\n[OK] Pendaftaran berhasil. Nomor RM: {no_rm}")
    return no_rm


# ============================================================
# MODUL ANTRIAN  (FR-3, FR-4, FR-5)
# ============================================================

def tambah_ke_antrian(no_rm, kategori):
    """
    Menambahkan pasien ke antrian sesuai kategori prioritasnya.
    Parameters:
        no_rm (str): Nomor RM pasien yang akan ditambahkan ke antrian
        kategori (str): Kategori prioritas pasien ("darurat"/"lansia"/"umum")
    Returns:
        None: Tidak mengembalikan nilai -- langsung menambah no_rm ke
        list antrian[kategori]. Kompleksitas O(1).
    """
    antrian[kategori].append(no_rm)


def panggil_pasien_berikutnya():
    """
    Memanggil pasien berikutnya berdasarkan urutan prioritas:
    darurat -> lansia -> umum, FIFO di dalam tiap kategori.
    Kompleksitas O(n) karena pop(0) menggeser elemen list.
    Parameters:
        Tidak ada.
    Returns:
        str atau None: Nomor RM pasien yang dipanggil, None jika semua
        antrian (darurat/lansia/umum) kosong
    """
    print("\n=== PANGGIL PASIEN BERIKUTNYA ===")

    for kategori in URUTAN_PRIORITAS:
        if antrian[kategori]:
            no_rm = antrian[kategori].pop(0)
            info = data_pasien[no_rm]

            print(f"\n>> Memanggil dari antrian: {kategori.upper()}")
            print("-" * 34)
            print(f"NOMOR RM  : {no_rm}")
            print(f"NAMA      : {info['nama']}")
            print(f"KATEGORI  : {label_kategori(kategori)}")
            print("-" * 34)

            catat_layanan_selesai(no_rm)
            return no_rm

    print("\nSemua antrian kosong — tidak ada pasien yang menunggu.")
    return None


def tampilkan_riwayat_singkat(no_rm):
    """
    Menampilkan ringkasan riwayat penyakit/diagnosa sebelumnya untuk satu
    pasien. Dipanggil otomatis dari daftar_kunjungan_ulang() (fitur 1,
    saat pasien lama terdeteksi) supaya petugas loket bisa langsung
    memberi tahu riwayat penyakit pasien -- membantu penanganan awal
    sebelum pasien masuk ke ruang periksa. Data yang ditampilkan memakai
    struktur riwayat_kunjungan yang sudah ada -- tidak perlu struktur
    data baru.
    Parameters:
        no_rm (str): Nomor RM pasien yang riwayatnya akan ditampilkan
    Returns:
        None: Tidak mengembalikan nilai -- langsung mencetak ringkasan
        riwayat penyakit ke layar.
    """
    riwayat = riwayat_kunjungan.get(no_rm, [])

    if not riwayat:
        print("Riwayat penyakit sebelumnya : (belum ada, pasien baru)")
        return

    print("Riwayat penyakit sebelumnya:")
    for kunjungan in riwayat:
        tanggal_str = kunjungan["tanggal"].strftime("%d-%m-%Y")
        print(f"  - {kunjungan['diagnosa']} ({kunjungan['poli']}, {tanggal_str})")


def catat_layanan_selesai(no_rm):
    """
    Mencatat durasi layanan dan riwayat kunjungan setelah pasien dipanggil.
    Parameters:
        no_rm (str): Nomor RM pasien yang baru selesai dilayani
    Returns:
        None: Tidak mengembalikan nilai -- langsung menambah durasi ke
        riwayat_durasi_layanan dan memanggil tambah_riwayat_kunjungan().
    """
    while True:
        teks_durasi = input("Masukkan durasi layanan (menit): ").strip()
        try:
            durasi = float(teks_durasi)
            if durasi <= 0:
                print("  [!] Durasi harus lebih dari 0.")
                continue
            break
        except ValueError:
            print("  [!] Durasi harus berupa angka.")

    riwayat_durasi_layanan.append(durasi)

    poli = input("Poli/loket yang menangani: ").strip() or "Umum"
    diagnosa = input("Diagnosa/catatan (boleh kosong): ").strip() or "-"
    tambah_riwayat_kunjungan(no_rm, poli, diagnosa)


def estimasi_waktu_tunggu(no_rm):
    """
    Menghitung estimasi waktu tunggu pasien berdasarkan posisinya dalam
    antrian dan rata-rata durasi layanan historis.
    Parameters:
        no_rm (str): Nomor RM pasien yang akan dihitung estimasi tunggunya
    Returns:
        float atau None: Estimasi waktu tunggu dalam menit, None jika
        Nomor RM tidak ditemukan atau pasien tidak sedang dalam antrian
    """
    if no_rm not in data_pasien:
        print("  [!] Nomor RM tidak ditemukan.")
        return None

    kategori = data_pasien[no_rm]["kategori"]

    if no_rm not in antrian[kategori]:
        print("  [!] Pasien tidak sedang dalam antrian (mungkin sudah dipanggil).")
        return None

    posisi = antrian[kategori].index(no_rm) + 1

    if kategori == "lansia":
        posisi += len(antrian["darurat"])
    elif kategori == "umum":
        posisi += len(antrian["darurat"]) + len(antrian["lansia"])

    if riwayat_durasi_layanan:
        rata_rata = sum(riwayat_durasi_layanan) / len(riwayat_durasi_layanan)
    else:
        rata_rata = WAKTU_DEFAULT

    estimasi = posisi * rata_rata

    print("\n=== ESTIMASI WAKTU TUNGGU ===")
    print(f"Nama       : {data_pasien[no_rm]['nama']}")
    print(f"Kategori   : {kategori.capitalize()}")
    print(f"Posisi     : ke-{posisi} dalam antrian")
    print(f"Estimasi   : ± {estimasi:.0f} menit")
    return estimasi


def tampilkan_antrian():
    """
    Menampilkan seluruh antrian saat ini, dikelompokkan per kategori.
    Parameters:
        Tidak ada.
    Returns:
        None: Tidak mengembalikan nilai -- langsung mencetak daftar
        antrian ke layar.
    """
    print("\n=== ANTRIAN SAAT INI ===")
    total = 0

    for kategori in URUTAN_PRIORITAS:
        daftar = antrian[kategori]
        print(f"\n{label_kategori(kategori)} ({len(daftar)} pasien)")
        if not daftar:
            print("  (kosong)")
        for i, no_rm in enumerate(daftar, 1):
            info = data_pasien[no_rm]
            print(f"  {i}. {no_rm} - {info['nama']} ({info['umur']} th)")
        total += len(daftar)

    print(f"\nTotal pasien menunggu: {total}")


# ============================================================
# MODUL RIWAYAT  (FR-6, FR-7)
# ============================================================

def tambah_riwayat_kunjungan(no_rm, poli, diagnosa):
    """
    Menambahkan satu entri riwayat kunjungan untuk pasien tertentu.
    Parameters:
        no_rm (str): Nomor RM pasien
        poli (str): Nama poli/loket yang menangani kunjungan ini
        diagnosa (str): Diagnosa atau catatan medis untuk kunjungan ini
    Returns:
        None: Tidak mengembalikan nilai -- langsung menambah entri baru
        ke riwayat_kunjungan[no_rm].
    """
    if no_rm not in riwayat_kunjungan:
        riwayat_kunjungan[no_rm] = []

    riwayat_kunjungan[no_rm].append({
        "tanggal": datetime.now(),
        "poli": poli,
        "diagnosa": diagnosa,
    })


def cari_no_rm_pasien_lama():
    """
    Mencari no_rm pasien LAMA -- dipakai bersama oleh fitur 1 (kunjungan
    ulang, supaya tidak perlu daftar biodata dari nol) dan fitur 5
    (riwayat kunjungan), supaya logika pencarian tidak diduplikasi.

    Mendukung 3 cara: Nomor RM (O(1), tapi sulit diingat), NIK (unik per
    orang, aman dari kesamaan nama), atau Nama (mudah diingat, tapi bisa
    lebih dari satu orang punya nama sama -- makanya hasilnya diminta
    dipilih ulang).
    Parameters:
        Tidak ada.
    Returns:
        str atau None: Nomor RM pasien jika ditemukan (atau berhasil
        dipilih saat ada nama ganda), None jika tidak ditemukan
    """
    print("Cari pasien berdasarkan:")
    print("  1. Nomor RM")
    print("  2. NIK")
    print("  3. Nama")
    pilihan = input("Pilih (1-3): ").strip()

    if pilihan == "1":
        no_rm = input("Masukkan Nomor RM: ").strip().upper()
        return no_rm if no_rm in data_pasien else None

    elif pilihan == "2":
        nik = input("Masukkan NIK (16 digit): ").strip()
        for no_rm, info in data_pasien.items():
            if info["nik"] == nik:
                return no_rm
        return None

    elif pilihan == "3":
        kata_kunci = input("Masukkan nama (atau sebagian nama): ").strip().lower()
        hasil = [
            no_rm for no_rm, info in data_pasien.items()
            if kata_kunci in info["nama"].lower()
        ]

        if len(hasil) == 1:
            return hasil[0]

        if len(hasil) > 1:
            print(f"\nDitemukan {len(hasil)} pasien dengan nama serupa:")
            for no_rm in hasil:
                info = data_pasien[no_rm]
                print(f"  {no_rm} - {info['nama']} ({info['umur']} th, NIK: {info['nik']})")
            pilihan_rm = input("Masukkan Nomor RM yang dimaksud: ").strip().upper()
            return pilihan_rm if pilihan_rm in hasil else None

        return None

    else:
        print("  [!] Pilihan tidak valid.")
        return None


def tampilkan_riwayat():
    """
    Menampilkan riwayat kunjungan seorang pasien. Pasien dicari lewat
    cari_no_rm_pasien_lama() supaya pengguna tidak wajib mengingat
    Nomor RM -- bisa juga lewat NIK atau nama.
    Parameters:
        Tidak ada.
    Returns:
        None: Tidak mengembalikan nilai -- langsung mencetak riwayat
        kunjungan ke layar.
    """
    print("\n=== RIWAYAT KUNJUNGAN ===")
    no_rm = cari_no_rm_pasien_lama()

    if no_rm is None:
        print("  [!] Pasien tidak ditemukan.")
        return

    nama = data_pasien[no_rm]["nama"]
    riwayat = riwayat_kunjungan.get(no_rm, [])

    print(f"\nRiwayat kunjungan {nama} ({no_rm}):")
    if not riwayat:
        print("  Belum ada riwayat kunjungan.")
        return

    print(f"{'No':<4}{'Tanggal':<20}{'Poli':<15}{'Diagnosa'}")
    print("-" * 60)
    for i, kunjungan in enumerate(riwayat, 1):
        tanggal_str = kunjungan["tanggal"].strftime("%d-%m-%Y %H:%M")
        print(f"{i:<4}{tanggal_str:<20}{kunjungan['poli']:<15}{kunjungan['diagnosa']}")

    print(f"\nTotal kunjungan: {len(riwayat)}")


def tampilkan_detail_pasien(no_rm, info):
    """
    Menampilkan detail satu pasien.
    Parameters:
        no_rm (str): Nomor RM pasien
        info (dict): Data biodata pasien (nama, umur, kategori, keluhan,
            waktu_daftar)
    Returns:
        None: Tidak mengembalikan nilai -- langsung mencetak kartu detail
        pasien ke layar.
    """
    print("-" * 40)
    print(f"{no_rm} - {info['nama']}")
    print(f"Umur: {info['umur']} | Kategori: {info['kategori'].capitalize()}")
    print(f"Keluhan   : {info['keluhan']}")
    print(f"Terdaftar : {info['waktu_daftar'].strftime('%d-%m-%Y %H:%M')}")
    print("-" * 40)


def cari_pasien():
    """
    Mencari pasien berdasarkan nomor RM (akses langsung, O(1))
    atau nama (linear search, O(n)).
    Parameters:
        Tidak ada.
    Returns:
        None: Tidak mengembalikan nilai -- langsung mencetak hasil
        pencarian ke layar.
    """
    print("\n=== CARI PASIEN ===")
    print("Cari berdasarkan:")
    print("  1. Nomor RM")
    print("  2. Nama")
    pilihan = input("Pilih (1-2): ").strip()

    if pilihan == "1":
        no_rm = input("Masukkan Nomor RM: ").strip().upper()
        if no_rm in data_pasien:
            tampilkan_detail_pasien(no_rm, data_pasien[no_rm])
        else:
            print("  [!] Pasien tidak ditemukan.")

    elif pilihan == "2":
        kata_kunci = input("Masukkan nama (atau sebagian nama): ").strip().lower()
        hasil = [
            (no_rm, info) for no_rm, info in data_pasien.items()
            if kata_kunci in info["nama"].lower()
        ]

        if not hasil:
            print("  [!] Tidak ada pasien ditemukan.")
            return

        print(f"\nDitemukan {len(hasil)} hasil:")
        for no_rm, info in hasil:
            tampilkan_detail_pasien(no_rm, info)

    else:
        print("  [!] Pilihan tidak valid.")


# ============================================================
# MODUL LAPORAN  (FR-8, FR-9)
# ============================================================

def tampilkan_statistik_harian():
    """
    Menampilkan jumlah pasien per kategori dan rata-rata waktu tunggu
    hari ini.
    Parameters:
        Tidak ada.
    Returns:
        None: Tidak mengembalikan nilai -- langsung mencetak statistik
        ke layar.
    """
    print("\n=== STATISTIK HARIAN ===")
    hari_ini = datetime.now().date()

    hitung = {"darurat": 0, "lansia": 0, "umum": 0}
    for info in data_pasien.values():
        if info["waktu_daftar"].date() == hari_ini:
            hitung[info["kategori"]] += 1

    print(f"{'Kategori':<15}| Jumlah Pasien")
    print("-" * 30)
    for kategori in URUTAN_PRIORITAS:
        print(f"{kategori.capitalize():<15}| {hitung[kategori]}")
    print("-" * 30)
    print(f"{'TOTAL':<15}| {sum(hitung.values())}")

    if riwayat_durasi_layanan:
        rata_rata = sum(riwayat_durasi_layanan) / len(riwayat_durasi_layanan)
        print(f"\nRata-rata waktu tunggu: {rata_rata:.1f} menit")
    else:
        print("\nBelum ada data waktu layanan.")


def urutkan_laporan():
    """
    Menampilkan laporan seluruh pasien, diurutkan sesuai kriteria
    pilihan pengguna (waktu daftar, kategori, atau jumlah kunjungan).
    Parameters:
        Tidak ada.
    Returns:
        None: Tidak mengembalikan nilai -- langsung mencetak laporan
        terurut ke layar.
    """
    print("\n=== LAPORAN KUNJUNGAN ===")

    if not data_pasien:
        print("  Belum ada data pasien.")
        return

    print("Urutkan berdasarkan:")
    print("  1. Waktu daftar")
    print("  2. Kategori")
    print("  3. Jumlah kunjungan")
    pilihan = input("Pilih (1-3): ").strip()

    daftar = [{"no_rm": no_rm, **info} for no_rm, info in data_pasien.items()]

    if pilihan == "1":
        hasil = sorted(daftar, key=lambda x: x["waktu_daftar"])
    elif pilihan == "2":
        urutan_kategori = {"darurat": 0, "lansia": 1, "umum": 2}
        hasil = sorted(daftar, key=lambda x: urutan_kategori[x["kategori"]])
    elif pilihan == "3":
        hasil = sorted(
            daftar,
            key=lambda x: len(riwayat_kunjungan.get(x["no_rm"], [])),
            reverse=True,
        )
    else:
        print("  [!] Pilihan tidak valid.")
        return

    print(f"\n{'No':<4}{'No.RM':<8}{'Nama':<20}{'Kategori':<10}{'Waktu Daftar'}")
    print("-" * 65)
    for i, p in enumerate(hasil, 1):
        waktu_str = p["waktu_daftar"].strftime("%H:%M")
        print(f"{i:<4}{p['no_rm']:<8}{p['nama']:<20}{p['kategori'].capitalize():<10}{waktu_str}")


# ============================================================
# MENU UTAMA
# ============================================================

def tampilkan_menu():
    """
    Menampilkan kotak menu utama sesuai desain UI yang disepakati.
    Parameters:
        Tidak ada.
    Returns:
        None: Tidak mengembalikan nilai -- langsung mencetak kotak menu
        ke layar.
    """
    print("        SISTEM ANTRIAN RUMAH SAKIT")
    print("     Algoritma dan Pemrograman — 2026")
    print("╔══════════════════════════════════════════╗")
    print("║         SISTEM ANTRIAN RUMAH SAKIT       ║")
    print("╠══════════════════════════════════════════╣")
    print("║  1. Daftar Pasien Baru                   ║")
    print("║  2. Panggil Pasien Berikutnya            ║")
    print("║  3. Lihat Antrian Saat Ini               ║")
    print("║  4. Cari Pasien                          ║")
    print("║  5. Riwayat Kunjungan Pasien             ║")
    print("║  6. Statistik Harian                     ║")
    print("║  7. Laporan (Urutkan Data)               ║")
    print("║  8. Estimasi Waktu Tunggu                ║")
    print("║  0. Keluar                               ║")
    print("╠══════════════════════════════════════════╣")


def main_menu():
    """
    Menampilkan menu utama dan menjalankan loop program.
    Parameters:
        Tidak ada.
    Returns:
        None: Tidak mengembalikan nilai -- loop berjalan sampai pengguna
        memilih menu "0" (Keluar).
    """
    aksi = {
        "1": daftar_pasien,
        "2": panggil_pasien_berikutnya,
        "3": tampilkan_antrian,
        "4": cari_pasien,
        "5": tampilkan_riwayat,
        "6": tampilkan_statistik_harian,
        "7": urutkan_laporan,
        "8": lambda: estimasi_waktu_tunggu(input("Masukkan Nomor RM: ").strip().upper()),
    }

    while True:
        tampilkan_menu()
        pilihan = input("║  Pilih menu (0-8): ").strip()
        print("╚══════════════════════════════════════════╝")

        if pilihan == "0":
            print("\nTerima kasih telah menggunakan sistem ini.")
            print("Wassalamu'alaikum Warahmatullahi Wabarakatuh.")
            break

        if pilihan in aksi:
            aksi[pilihan]()
        else:
            print("  [!] Pilihan tidak valid! Silakan coba lagi.")


# Jalankan program
if __name__ == "__main__":
    muat_data_demo()
    main_menu()


        SISTEM ANTRIAN RUMAH SAKIT
     Algoritma dan Pemrograman — 2026
╔══════════════════════════════════════════╗
║         SISTEM ANTRIAN RUMAH SAKIT       ║
╠══════════════════════════════════════════╣
║  1. Daftar Pasien Baru                   ║
║  2. Panggil Pasien Berikutnya            ║
║  3. Lihat Antrian Saat Ini               ║
║  4. Cari Pasien                          ║
║  5. Riwayat Kunjungan Pasien             ║
║  6. Statistik Harian                     ║
║  7. Laporan (Urutkan Data)               ║
║  8. Estimasi Waktu Tunggu                ║
║  0. Keluar                               ║
╠══════════════════════════════════════════╣
║  Pilih menu (0-8): 5
╚══════════════════════════════════════════╝

=== RIWAYAT KUNJUNGAN ===
Cari pasien berdasarkan:
  1. Nomor RM
  2. NIK
  3. Nama
Pilih (1-3): 1
Masukkan Nomor RM: RM004

Riwayat kunjungan Sahid Haqqi (RM004):
No  Tanggal             Poli           Diagnosa
------------------------------------------------------------
